In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:

data_path = os.path.join(path, 'Q3_data.csv')
df_data = pd.read_csv(data_path)

print(f"Shape: {df_data.shape}")

In [ ]:
# Task 2: Write your code here:

df_data.head()

In [ ]:
# Task 3: Write your code here:

df_data.info()

In [ ]:
# Task 4: Write your code here:

df_data.describe()

In [ ]:
# Task 1: Write your code here:

# Missing values
print("Missing values:")
print(df_data.isnull().sum())

In [ ]:
stat_cols = ['P_2', 'B_2', 'D_142', 'D_143','D_144','D_145']

# Drop rows with missing stat values
df_clean = df_data.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")

# Fill missing delivery time
df_clean['Target'] = df_clean['Target'].fillna('none')

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) ##inplace changes original data
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_data)

In [ ]:
# Task 3: Write your code here:

# Encode features and target using LabelEncoder
categorical_cols = df_data.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_data[col] = le.fit_transform(df_data[col])
df_data

In [ ]:
# Task 4: Write your code here:

# Standardize features using StandardScaler
numerical_cols = df_data.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df_data[numerical_cols] = scaler.fit_transform(df_data[numerical_cols])
df_data

In [ ]:
# Task 5: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_data, "Target")

#not imbalanced

In [ ]:
# Task 1: Write your code here:

X = df_data.drop("Target", axis=1).astype(float) ##should be same types of could raise errors in the model when writing the equation in code, in SKlearn we dont need to do it but in numpy u cant do math betwwen 2 different types
y = df_data['Target'].astype(float)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier

In [ ]:
model= CatBoostClassifier( ##for each level asks question , catgorical columns no need for encoding
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

In [ ]:
# Task 2,3,4,5: Write your code here:

# sigmoid in NumPy
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

# BCE in NumPy
def binary_cross_entropy(y, y_hat):
  epsilon = 1e-15  # Very small number to prevent log(0)
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # np.clip(value, min, max) ##if its out of range clip this certain value

  loss = -1/len(y) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
  return loss


def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape # m rows, n columns (dimensions)
  theta = np.zeros(n) # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Logistic Regression"):
    z = np.dot(X, theta)
    y_hat = sigmoid(z)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = binary_cross_entropy(y, y_hat)
    losses.append(loss)

  return theta, losses

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)



  print(f"  Accuracy:  {np.mean(accuracy[model]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(f1[model]['f1']):.4f}")


In [ ]:
# Task 1: Write your code here:

# Gather importances from the models (from the last fold)
importances = {}


importances['CatBoost'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

##the top 1 with longest figure im guessing D4 (i see there's something wrong in my looping)

In [ ]:
# Task Bonus: Write your code here: